# 01 — Corpus Inspection

## Research question

What linguistic material is available in the curated conversational corpus, and how is it organised for computational analysis?

## Objectives

1. Load and inspect the cleaned conversational corpus.
2. Verify the number of exchanges and speaker turns.
3. Examine the structure and content of the conversational data.
4. Identify occurrences of the target creative expressions.
5. Establish a descriptive baseline for subsequent NLP analyses.

## Data and methods

The analysis uses `data/public/conversations_adapted.json`, a structured dataset containing ten selected user–assistant exchanges.

The public-facing [Adapted Exchange Guide](../docs/corpus/adapted_exchanges.md) provides contextual descriptions but is not used as the analytical dataset.

This initial notebook focuses on corpus inspection and descriptive analysis rather than statistical inference.

## 1. Structural inspection

We begin by loading the cleaned conversational corpus and examining its structure.

The objective is to verify how exchanges and speaker turns are represented before transforming the data for analysis.

In [1]:
import json
from pathlib import Path

# Define the path to the cleaned corpus
corpus_path = Path("../data/public/conversations_adapted.json")

# Load the JSON file
with corpus_path.open("r", encoding="utf-8") as file:
    corpus = json.load(file)

# Inspect the top-level structure
print("Data type:", type(corpus))
print("Number of exchanges:", len(corpus))

Data type: <class 'list'>
Number of exchanges: 10


In [2]:
# Inspect the structure of the first exchange
first_exchange = corpus[0]

print("Exchange fields:", list(first_exchange.keys()))
print("Exchange ID:", first_exchange["exchange_id"])
print("Number of turns:", len(first_exchange["turns"]))

print("\nSpeaker turns:")
for turn in first_exchange["turns"]:
    print(f"- {turn['speaker']}: {len(turn['text'])} characters")

Exchange fields: ['exchange_id', 'title', 'source_type', 'turns']
Exchange ID: EX001
Number of turns: 2

Speaker turns:
- user: 502 characters
- assistant: 231 characters


In [3]:
# Inspect the structure of every exchange

for exchange in corpus:
    exchange_id = exchange["exchange_id"]
    turns = exchange["turns"]

    print(f"{exchange_id}: {len(turns)} turns")

EX001: 2 turns
EX002: 2 turns
EX003: 2 turns
EX004: 2 turns
EX005: 2 turns
EX006: 2 turns
EX007: 2 turns
EX008: 2 turns
EX009: 2 turns
EX010: 2 turns


## 2. Corpus integrity

We verify that exchange identifiers are unique, that each exchange
contains the expected speaker turns, and that the text fields contain
valid, non-empty strings.

These checks complement the validation performed during corpus preparation.

In [4]:
# Check exchange identifiers
exchange_ids = [exchange["exchange_id"] for exchange in corpus]

print("Unique exchange IDs:", len(set(exchange_ids)))
print("Duplicate IDs:", len(exchange_ids) - len(set(exchange_ids)))

# Check speaker turns and text fields
for exchange in corpus:
    exchange_id = exchange["exchange_id"]
    turns = exchange["turns"]

    speakers = [turn["speaker"] for turn in turns]

    valid_speakers = speakers == ["user", "assistant"]

    valid_text = all(
        isinstance(turn["text"], str) and bool(turn["text"].strip())
        for turn in turns
    )

    print(
        f"{exchange_id}: "
        f"speakers valid = {valid_speakers}, "
        f"text valid = {valid_text}"
    )

Unique exchange IDs: 10
Duplicate IDs: 0
EX001: speakers valid = True, text valid = True
EX002: speakers valid = True, text valid = True
EX003: speakers valid = True, text valid = True
EX004: speakers valid = True, text valid = True
EX005: speakers valid = True, text valid = True
EX006: speakers valid = True, text valid = True
EX007: speakers valid = True, text valid = True
EX008: speakers valid = True, text valid = True
EX009: speakers valid = True, text valid = True
EX010: speakers valid = True, text valid = True


## 3. Descriptive profiling

We examine the distribution of linguistic material across exchanges and speakers.

Our initial measures include:

- Number of conversational turns.
- Character counts per turn.
- Approximate word counts per turn.
- Total contributions by speaker.

These descriptive measures establish the size and composition of the corpus before lexical analysis.

In [5]:
import pandas as pd

# Convert nested conversational data into a tabular structure
records = []

for exchange in corpus:
    for turn in exchange["turns"]:
        records.append({
            "exchange_id": exchange["exchange_id"],
            "speaker": turn["speaker"],
            "text": turn["text"]
        })

df_turns = pd.DataFrame(records)

print("DataFrame shape:", df_turns.shape)
df_turns.head()

DataFrame shape: (20, 3)


,exchange_id,speaker,text
0,EX001,user,"The combined model didn't work, so we sliced t..."
1,EX001,assistant,"HAHAHA. Go eat and shower, because you have ju..."
2,EX002,user,I've just finished editing changes in 19 EDA n...
3,EX002,assistant,SHE'S BACK!\n\nYou didn't come back from Greec...
4,EX003,user,Maria Obsesiones' unicorning in environmentall...


In [6]:
# Calculate descriptive measures for each turn

df_turns["char_count"] = df_turns["text"].str.len()

df_turns["word_count"] = (
    df_turns["text"]
    .str.split()
    .str.len()
)

df_turns[
    ["exchange_id", "speaker", "char_count", "word_count"]
]

,exchange_id,speaker,char_count,word_count
0,EX001,user,502,93
1,EX001,assistant,231,39
2,EX002,user,232,44
3,EX002,assistant,456,67
4,EX003,user,1120,182
5,EX003,assistant,387,53
6,EX004,user,298,49
7,EX004,assistant,819,108
8,EX005,user,412,77
9,EX005,assistant,676,115


In [7]:
# Summarise corpus size by speaker

speaker_summary = (
    df_turns
    .groupby("speaker")
    .agg(
        turns=("text", "count"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
        mean_words_per_turn=("word_count", "mean")
    )
    .round(2)
)

speaker_summary

,turns,total_characters,total_words,mean_words_per_turn
speaker,,,,
assistant,10,5594,817,81.7
user,10,4136,715,71.5


In [8]:
# Overall corpus size

print("Total exchanges:", df_turns["exchange_id"].nunique())
print("Total turns:", len(df_turns))
print("Total characters:", df_turns["char_count"].sum())
print("Approximate total words:", df_turns["word_count"].sum())

Total exchanges: 10
Total turns: 20
Total characters: 9730
Approximate total words: 1532


### Descriptive results

The curated corpus contains 10 exchanges comprising 20 speaker-labelled text excerpts, with 10 contributions attributed to each speaker.

The dataset contains 12,902 characters and approximately 2,056 whitespace-separated words.

The assistant excerpts contribute 1,161 words (56.5%), compared with 895 words (43.5%) from the user excerpts.

Mean excerpt length is 116.1 words for the assistant and 89.5 words for the user.

These measurements describe the curated excerpts, not complete conversational turns. The source material was manually selected to preserve relevant linguistic phenomena and their surrounding context. Consequently, differences in excerpt length and speaker contribution may reflect editorial selection and should not be interpreted as differences in conversational behaviour.

Word counts are preliminary and will be revisited during tokenisation analysis in Notebook 02.

## 4. Linguistic inspection

This section identifies occurrences of creative linguistic expressions
and examines their distribution across exchanges and speakers.

The initial inspection focuses on the Frankenstein lexical family.

Computational identification of expressions is distinguished from
qualitative interpretation of their morphological, syntactic,
semantic and metalinguistic functions.

In [9]:
# Identify turns containing the Frankenstein lexical family

frankenstein_turns = df_turns[
    df_turns["text"].str.contains(
        "frankenstein",
        case=False,
        regex=False
    )
]

frankenstein_turns[
    ["exchange_id", "speaker", "word_count"]
]

,exchange_id,speaker,word_count
0,EX001,user,93
3,EX002,assistant,67
4,EX003,user,182
5,EX003,assistant,53
6,EX004,user,49
7,EX004,assistant,108
9,EX005,assistant,115
11,EX006,assistant,106
13,EX007,assistant,38
15,EX008,assistant,118


### Preliminary observations

Forms containing the sequence *frankenstein* occur in 15 of the
20 speaker-labelled excerpts, distributed across all 10 exchanges.

The user contributes such forms in 4 exchanges, while the assistant
contributes them in 9 exchanges.

In 6 exchanges (EX002, EX005, EX006, EX007, EX008 and EX010),
the target sequence appears in the assistant excerpt but not in
the corresponding user excerpt.

This provides evidence of lexical reuse without an occurrence of
the target sequence in the paired user excerpt.

However, absence of the sequence does not imply absence of
conversational prompting. Interpretation of reuse and extension
requires examination of the surrounding linguistic context.

These observations concern the curated corpus and do not establish
general patterns of language-model behaviour.

In [10]:
# Count occurrences of the Frankenstein lexical family in each turn

df_turns["frankenstein_count"] = (
    df_turns["text"]
    .str.count("(?i)frankenstein")
)

df_turns.loc[
    df_turns["frankenstein_count"] > 0,
    ["exchange_id", "speaker", "frankenstein_count"]
]

,exchange_id,speaker,frankenstein_count
0,EX001,user,1
3,EX002,assistant,1
4,EX003,user,1
5,EX003,assistant,1
6,EX004,user,1
7,EX004,assistant,2
9,EX005,assistant,1
11,EX006,assistant,1
13,EX007,assistant,1
15,EX008,assistant,1


In [11]:
# Summarise target occurrences by speaker

occurrence_summary = (
    df_turns
    .groupby("speaker")
    .agg(
        turns_with_target=("frankenstein_count", lambda x: (x > 0).sum()),
        total_occurrences=("frankenstein_count", "sum")
    )
)

occurrence_summary

,turns_with_target,total_occurrences
speaker,,
assistant,9,10
user,4,4


In [12]:
# Total occurrences across the curated corpus

print(
    "Total Frankenstein-family occurrences:",
    df_turns["frankenstein_count"].sum()
)

Total Frankenstein-family occurrences: 14


In [13]:
# Extract word-like forms containing "frankenstein"

pattern = r"\b[\w-]*frankenstein[\w-]*\b"

df_turns["frankenstein_forms"] = (
    df_turns["text"]
    .str.findall(pattern, flags=2)
)

df_turns.loc[
    df_turns["frankenstein_forms"].str.len() > 0,
    ["exchange_id", "speaker", "frankenstein_forms"]
]

,exchange_id,speaker,frankenstein_forms
0,EX001,user,[Frankenstein]
3,EX002,assistant,[Frankenstein]
4,EX003,user,[frankensteining]
5,EX003,assistant,[Frankenstein]
6,EX004,user,[frankensteining]
7,EX004,assistant,"[Frankensteining-Unicorning, Frankensteining]"
9,EX005,assistant,[Frankenstein]
11,EX006,assistant,[super-Frankenstein]
13,EX007,assistant,[Frankenstein]
15,EX008,assistant,[Frankenstein]


### Preliminary form inventory

The search identifies 14 occurrences of the *Frankenstein*
lexical family across the curated corpus.

Three distinct word-like forms are observed when case differences
are ignored: *Frankenstein*, *Frankensteining*, and
*super-Frankenstein*.

This inventory describes orthographic forms rather than linguistic
constructions. Identical forms may occur in different syntactic
environments and support different semantic or metaphorical uses.

Contextual inspection is therefore required before assigning
linguistic categories.

### Concordance analysis (KWIC)

A Key Word in Context (KWIC) concordance is used to examine the
immediate linguistic environment of each target occurrence.

An initial window of 80 characters before and after each match
is selected for exploratory inspection.

This window is intended to facilitate comparison across occurrences,
not to define the complete context required for semantic or
pragmatic interpretation.

In [14]:
import re

# Define the target pattern and context window
pattern = re.compile(r"frankenstein", flags=re.IGNORECASE)
window = 80

# Store concordance records
kwic_records = []

for _, row in df_turns.iterrows():
    text = row["text"]

    for match in pattern.finditer(text):
        start, end = match.span()

        kwic_records.append({
            "exchange_id": row["exchange_id"],
            "speaker": row["speaker"],
            "left_context": text[max(0, start - window):start],
            "target": text[start:end],
            "right_context": text[end:end + window]
        })

df_kwic = pd.DataFrame(kwic_records)

print("Concordance entries:", len(df_kwic))
df_kwic

Concordance entries: 14


,exchange_id,speaker,left_context,target,right_context
0,EX001,user,"actually pruned a lot of it, made Russian doll...",Frankenstein,"it. And it worked, and it was epic!\n\nOK, I ..."
1,EX002,assistant,d every artefact and apparently cleaned 39 not...,Frankenstein,architecture makes sense.\n\nLast time you ha...
2,EX003,user,siones' unicorning in environmentally friendly...,frankenstein,"ing, translated into human lingo, basically. 🌈..."
3,EX003,assistant,sentences back.\n\nEnvironmentally friendly r...,Frankenstein,disposal. 🌈🦄♻️
4,EX004,user,I need to change the zombie roadmap that turn...,frankenstein,"ing and the unicorning, because of the finalís..."
5,EX004,assistant,this gives the zombie-guide a better ending b...,Frankenstein,ing-Unicorning logic rather than merely listin...
6,EX004,assistant,Frankensteining-Unicorning logic rather than ...,Frankenstein,"ing: Reconstruct the original experiment, pres..."
7,EX005,assistant,'re allowed to notice it. You're not allowed t...,Frankenstein,phase took four months because we didn't know...
8,EX006,assistant,we'd end up back where we started:\n\n“Accord...,Frankenstein,", Agency has an average coefficient of 0.1837—..."
9,EX007,assistant,5 notebooks staring at us menacingly: unfortun...,Frankenstein,"lab, amigo. 😂"


### Complete-form concordance

The initial KWIC analysis identified occurrences of the character
sequence *frankenstein*.

We now extend the matching pattern to capture complete word-like
forms, including morphological derivatives and hyphenated compounds.

This allows us to distinguish the target forms from their immediate
linguistic environments while preserving the same 80-character
context window.

In [15]:
# Extract complete word-like forms with their surrounding context

form_pattern = re.compile(
    r"\b[\w-]*frankenstein[\w-]*\b",
    flags=re.IGNORECASE
)

form_kwic_records = []

for _, row in df_turns.iterrows():
    text = row["text"]

    for match in form_pattern.finditer(text):
        start, end = match.span()

        form_kwic_records.append({
            "exchange_id": row["exchange_id"],
            "speaker": row["speaker"],
            "left_context": text[max(0, start - window):start],
            "form": match.group(),
            "right_context": text[end:end + window]
        })

df_form_kwic = pd.DataFrame(form_kwic_records)

print("Complete-form concordance entries:", len(df_form_kwic))

df_form_kwic

Complete-form concordance entries: 14


,exchange_id,speaker,left_context,form,right_context
0,EX001,user,"actually pruned a lot of it, made Russian doll...",Frankenstein,"it. And it worked, and it was epic!\n\nOK, I ..."
1,EX002,assistant,d every artefact and apparently cleaned 39 not...,Frankenstein,architecture makes sense.\n\nLast time you ha...
2,EX003,user,siones' unicorning in environmentally friendly...,frankensteining,", translated into human lingo, basically. 🌈😁 T..."
3,EX003,assistant,sentences back.\n\nEnvironmentally friendly r...,Frankenstein,disposal. 🌈🦄♻️
4,EX004,user,I need to change the zombie roadmap that turn...,frankensteining,"and the unicorning, because of the finalísimo..."
5,EX004,assistant,this gives the zombie-guide a better ending b...,Frankensteining-Unicorning,logic rather than merely listing notebook num...
6,EX004,assistant,Frankensteining-Unicorning logic rather than ...,Frankensteining,": Reconstruct the original experiment, preserv..."
7,EX005,assistant,'re allowed to notice it. You're not allowed t...,Frankenstein,phase took four months because we didn't know...
8,EX006,assistant,.\n\nAnd we'd end up back where we started:\n\...,super-Frankenstein,", Agency has an average coefficient of 0.1837—..."
9,EX007,assistant,5 notebooks staring at us menacingly: unfortun...,Frankenstein,"lab, amigo. 😂"


### Linguistic annotation

A preliminary annotation table is prepared to examine the
constructional, syntactic, semantic and discourse-level properties
of the target expressions.

Each occurrence constitutes one annotation unit.

Automatic extraction provides the occurrence metadata and local
context. Linguistic categories will be assigned manually after
examining the relevant conversational material.

The annotation dimensions are kept separate to avoid conflating
morphological form, grammatical function and contextual meaning.

In [16]:
# Prepare one annotation record per target occurrence

df_annotations = df_form_kwic.copy()

annotation_columns = [
    "construction",
    "syntactic_function",
    "semantic_reference",
    "discourse_function",
    "preceding_user_lexical_cue",
    "contextual_semantic_cue",
    "cue_evidence"
]

for column in annotation_columns:
    df_annotations[column] = pd.NA

In [17]:
# Display the full concordance without truncating the text

with pd.option_context("display.max_colwidth", None):
    display(
        df_annotations[
            [
                "exchange_id",
                "speaker",
                "left_context",
                "form",
                "right_context"
            ]
        ]
    )

,exchange_id,speaker,left_context,form,right_context
0,EX001,user,"actually pruned a lot of it, made Russian dolls with it, but then we decided to",Frankenstein,"it. And it worked, and it was epic!\n\nOK, I need lunch and a shower, amigo. But"
1,EX002,assistant,d every artefact and apparently cleaned 39 notebooks!\n\nThe analysis exists. The,Frankenstein,"architecture makes sense.\n\nLast time you had a thought, we ended up constructin"
2,EX003,user,siones' unicorning in environmentally friendly rainbow conditions after serious,frankensteining,", translated into human lingo, basically. 🌈😁 That's what we need to do!\n\nAnd the"
3,EX003,assistant,sentences back.\n\nEnvironmentally friendly rainbow unicorning after responsible,Frankenstein,disposal. 🌈🦄♻️
4,EX004,user,I need to change the zombie roadmap that turned into a guide to explaining the,frankensteining,"and the unicorning, because of the finalísimo's numbering before the appendices"
5,EX004,assistant,this gives the zombie-guide a better ending because it can finally explain the,Frankensteining-Unicorning,logic rather than merely listing notebook numbers:\n\nFrankensteining: Reconstruc
6,EX004,assistant,Frankensteining-Unicorning logic rather than merely listing notebook numbers:\n\n,Frankensteining,": Reconstruct the original experiment, preserve the messy behavioural anatomy, s"
7,EX005,assistant,'re allowed to notice it. You're not allowed to follow it into the forest.\n\nThe,Frankenstein,phase took four months because we didn't know what the creature looked like. No
8,EX006,assistant,.\n\nAnd we'd end up back where we started:\n\n“According to the multimodel omnibus,super-Frankenstein,", Agency has an average coefficient of 0.1837—”\n\nCHEAP FOR WHAT?\n\nCHEAP FOR WHOM"
9,EX007,assistant,5 notebooks staring at us menacingly: unfortunately also ✅\n\nWelcome back to the,Frankenstein,"lab, amigo. 😂"


In [18]:
# Annotate EX001: original verbal innovation

df_annotations.loc[0, "construction"] = "Frankenstein it"

df_annotations.loc[0, "syntactic_function"] = "Transitive verb"

df_annotations.loc[0, "semantic_reference"] = (
    "Reconstruction of a complex analytical model"
)

df_annotations.loc[0, "discourse_function"] = (
    "Introduction of a playful metaphor for model reconstruction"
)

# Lexical and contextual cueing: not applicable to the user occurrence

df_annotations.loc[0, "preceding_user_lexical_cue"] = pd.NA

df_annotations.loc[0, "contextual_semantic_cue"] = pd.NA

df_annotations.loc[0, "cue_evidence"] = pd.NA

In [19]:
# Annotate EX002: Frankenstein architecture

# EX002 — Assistant: "Frankenstein architecture"

df_annotations.loc[1, "construction"] = "Frankenstein architecture"

df_annotations.loc[1, "syntactic_function"] = "Noun modifier"

df_annotations.loc[1, "semantic_reference"] = (
    "Reconstructed structure of the analytical model"
)

df_annotations.loc[1, "discourse_function"] = (
    "Reuse and extension of the established metaphor "
    "to describe the model's organisation"
)

df_annotations.loc[1, "preceding_user_lexical_cue"] = False

df_annotations.loc[1, "contextual_semantic_cue"] = "Present"

df_annotations.loc[1, "cue_evidence"] = (
    "The paired user excerpt does not contain a Frankenstein-family "
    "expression; the assistant applies the established reconstruction "
    "metaphor to the analytical model's architecture"
)

In [20]:
with pd.option_context("display.max_colwidth", None):
    display(df_annotations.loc[[0, 1]])

,exchange_id,speaker,left_context,form,right_context,construction,syntactic_function,semantic_reference,discourse_function,preceding_user_lexical_cue,contextual_semantic_cue,cue_evidence
0,EX001,user,"actually pruned a lot of it, made Russian dolls with it, but then we decided to",Frankenstein,"it. And it worked, and it was epic!\n\nOK, I need lunch and a shower, amigo. But",Frankenstein it,Transitive verb,Reconstruction of a complex analytical model,Introduction of a playful metaphor for model reconstruction,<NA>,<NA>,<NA>
1,EX002,assistant,d every artefact and apparently cleaned 39 notebooks!\n\nThe analysis exists. The,Frankenstein,"architecture makes sense.\n\nLast time you had a thought, we ended up constructin",Frankenstein architecture,Noun modifier,Reconstructed structure of the analytical model,Reuse and extension of the established metaphor to describe the model's organisation,False,Present,The paired user excerpt does not contain a Frankenstein-family expression; the assistant applies the established reconstruction metaphor to the analytical model's architecture


In [21]:
# Inspect both speaker excerpts from EX003

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX003",
            ["speaker", "text"]
        ]
    )

,speaker,text
4,user,"Maria Obsesiones' unicorning in environmentally friendly rainbow conditions after serious frankensteining, translated into human lingo, basically. 🌈😁 That's what we need to do!\n\nAnd then the README, because I want to explain the theoretical framework without turning the repository's introduction into a Bible without numbered verses.\n\nBy the way, read this from what I sent you:\n\nWhat Girotto and colleagues’ study claimed instead is that actors and readers produce different counterfactuals: while actors were more likely to alter the means or features by which they had ended up making a poor decision in the undoing of an event (that is, “If the means by which I made the wrong decision had been different, the outcome would have been different”), readers were more likely to undo someone else’s choice (that is, “If they had made a different choice, the outcome would have been different”).\n\nNow, isn’t it funny that our results show that when Focus = I, Agent = 0 and Goal = frequent, sentences lead to less transfer—and thus more correct responses?\n\nExcuses from actors even when they aren’t real actors! Amazing."
5,assistant,"In the beginning, Roese created the goal and Gibson the computational resource pool.\n\nThe README gives readers the human story. The opening notebook gives them the theory and experimental sentences. The analysis notebooks provide the evidence. The final notebook brings the actual sentences back.\n\nEnvironmentally friendly rainbow unicorning after responsible Frankenstein disposal. 🌈🦄♻️"


In [22]:
# EX003 — User: "after serious frankensteining"

df_annotations.loc[2, "construction"] = "after serious frankensteining"

df_annotations.loc[2, "syntactic_function"] = (
    "Nominal -ing form; head of a noun phrase"
)

df_annotations.loc[2, "semantic_reference"] = (
    "The analytical reconstruction process"
)

df_annotations.loc[2, "discourse_function"] = (
    "Characterisation of the analytical work as an intensive "
    "process preceding the final synthesis"
)

df_annotations.loc[2, "preceding_user_lexical_cue"] = pd.NA

df_annotations.loc[2, "contextual_semantic_cue"] = pd.NA

df_annotations.loc[2, "cue_evidence"] = pd.NA

In [23]:
# EX003 — Assistant: "after responsible Frankenstein disposal"

df_annotations.loc[3, "construction"] = (
    "after responsible Frankenstein disposal"
)

df_annotations.loc[3, "syntactic_function"] = (
    "Attributive noun modifier"
)

df_annotations.loc[3, "semantic_reference"] = (
    "Completion and management of the messy analytical "
    "reconstruction process, keeping its unnecessary "
    "complexity out of the reader-facing synthesis"
)

df_annotations.loc[3, "discourse_function"] = (
    "Constructional transformation of the user's metaphor, "
    "introducing a waste-management frame that connects "
    "the completed analytical work with the final synthesis"
)

df_annotations.loc[3, "preceding_user_lexical_cue"] = True

df_annotations.loc[3, "contextual_semantic_cue"] = "Present"

df_annotations.loc[3, "cue_evidence"] = (
    "User's 'after serious frankensteining' and "
    "'environmentally-friendly rainbow conditions'; "
    "assistant extends the process metaphor through "
    "'responsible Frankenstein disposal'"
)

In [24]:
# EX004 — Inspect both speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX004",
            ["speaker", "text"]
        ]
    )

,speaker,text
6,user,"Guess what? I need to change the zombie roadmap that turned into a guide to explaining the frankensteining and the unicorning, because of the finalísimo's numbering before the appendices!\n\nOh well. What is that for us now, after grading our guillotine skills? It did interrupt my ice cream, though!"
7,assistant,"THE ZOMBIE ROADMAP HAS RISEN AGAIN! 🧟‍♀️📜\n\nAfter what we've done, this is tiny surgery. No new intellectual work. It's repository plumbing.\n\nAnd actually, this gives the zombie-guide a better ending because it can finally explain the Frankensteining-Unicorning logic rather than merely listing notebook numbers:\n\nFrankensteining: Reconstruct the original experiment, preserve the messy behavioural anatomy, separate questions that shouldn't be averaged together, and build an auditable analytical path.\n\nUnicorning: Return the statistical patterns to the counterfactuals that generated them, reconnect them with the theoretical framework, and ask what they might mean for future psycholinguistic and second-language acquisition research.\n\nFinal synthesis: Where those two processes meet. Appendices: Technical receipts."


In [25]:
# EX004 — User: "explaining the frankensteining and the unicorning"

df_annotations.loc[4, "construction"] = (
    "explaining the frankensteining and the unicorning"
)

df_annotations.loc[4, "syntactic_function"] = (
    "Nominal -ing form; head of a coordinated noun phrase"
)

df_annotations.loc[4, "semantic_reference"] = (
    "The analytical reconstruction and modelling process "
    "documented in the modelling notebooks and appendices"
)

df_annotations.loc[4, "discourse_function"] = (
    "Identification of a process that the reader-facing "
    "guide needs to explain"
)

df_annotations.loc[4, "preceding_user_lexical_cue"] = pd.NA

df_annotations.loc[4, "contextual_semantic_cue"] = pd.NA

df_annotations.loc[4, "cue_evidence"] = pd.NA

In [26]:
# EX004 — Assistant: "Frankensteining-Unicorning logic"

df_annotations.loc[5, "construction"] = (
    "the Frankensteining-Unicorning logic"
)

df_annotations.loc[5, "syntactic_function"] = (
    "Hyphenated compound premodifier of the noun 'logic'"
)

df_annotations.loc[5, "semantic_reference"] = (
    "The established analytical reconstruction process "
    "as the first stage of the workflow"
)

df_annotations.loc[5, "discourse_function"] = (
    "Explicit representation of the established sequence "
    "from analytical reconstruction to findings synthesis"
)

df_annotations.loc[5, "preceding_user_lexical_cue"] = True

df_annotations.loc[5, "contextual_semantic_cue"] = "Present"

df_annotations.loc[5, "cue_evidence"] = (
    "User explicitly mentions frankensteining and unicorning "
    "and explains the need to update the guide after introducing "
    "the final synthesis notebook"
)

In [27]:
# EX004 — Assistant: metalinguistic definition

df_annotations.loc[6, "construction"] = (
    "Frankensteining: reconstruct the original experiment..."
)

df_annotations.loc[6, "syntactic_function"] = (
    "Nominal -ing form; metalinguistic entry label"
)

df_annotations.loc[6, "semantic_reference"] = (
    "The established process of experimental reconstruction, "
    "statistical analysis and development of an auditable "
    "analytical workflow"
)

df_annotations.loc[6, "discourse_function"] = (
    "Metalinguistic clarification of an established analytical "
    "process to support the organisation of the notebook guide"
)

df_annotations.loc[6, "preceding_user_lexical_cue"] = True

df_annotations.loc[6, "contextual_semantic_cue"] = "Present"

df_annotations.loc[6, "cue_evidence"] = (
    "User identifies the need to revise the guide's explanation "
    "of the analytical sequence following the introduction "
    "of the final synthesis notebook"
)

In [28]:
# EX005 — Inspect the paired speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX005",
            ["speaker", "text"]
        ]
    )

,speaker,text
8,user,"Maria Obsesiones is struggling with the mess and the typos. I'm about to demand 48-hour days and brew a massive pot of coffee while steam comes out of my ears.\n\nFirst I'm going to enjoy my mother's bolognese and some Greek-food memories. Then I'll return to my four-month project.\n\nFour months! And I still have the ML and NLP parts to do, and then I need to find a job! Help! I'm having a post-holiday crisis! 😂"
9,assistant,"Today's actual pipeline: Mother's bolognese → Greek-food nostalgia → coffee → open ONE notebook → edit what's already there.\n\nNo modelling. No new hypotheses because you notice an intriguing 0.07 somewhere. Absolutely NO UNICORN FLU. 🦄🚫\n\nAnything interesting that needs new analysis goes on an after-thesis list. You're allowed to notice it. You're not allowed to follow it into the forest.\n\nThe Frankenstein phase took four months because we didn't know what the creature looked like. Now we do.\n\nYou don't need another four months to discover it. You need to comb its hair, correct its spelling, put a jacket on it and make the README explain why it has bolts in its neck. 😂"


In [29]:
# EX005 — Assistant: "Frankenstein phase"

df_annotations.loc[7, "construction"] = (
    "Frankenstein phase"
)

df_annotations.loc[7, "syntactic_function"] = (
    "Attributive noun modifier of the head noun 'phase'"
)

df_annotations.loc[7, "semantic_reference"] = (
    "The completed intensive analytical reconstruction "
    "and modelling stage of the project"
)

df_annotations.loc[7, "discourse_function"] = (
    "Contrasts the demanding construction stage with "
    "the remaining editing work, providing reassurance "
    "about the project's progress"
)

df_annotations.loc[7, "preceding_user_lexical_cue"] = False

df_annotations.loc[7, "contextual_semantic_cue"] = "Present"

df_annotations.loc[7, "cue_evidence"] = (
    "User returns to a four-month project, expresses concern "
    "about the remaining work, and discusses the transition "
    "from construction to editing; no explicit Frankenstein "
    "expression appears in the paired user excerpt"
)

In [30]:
# EX006 — Inspect both speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX006",
            ["speaker", "text"]
        ]
    )

,speaker,text
10,user,"Maria Obsesiones is done with modelling!\n\nBut I must admit, the thought of multimodal modelling did cross my mind a few times, just out of curiosity.\n\nWhat bloody equation would the model come up with if it tried to average everything?\n\nIt would probably explode and refuse to cooperate! Hahaha."
11,assistant,"It might even converge. That's the dangerous part.\n\nIt could obediently hand you coefficients, standard errors and six interaction terms with absolutely magnificent decimal places while averaging over precisely the conditional structures we've spent months disentangling.\n\nThe equation wouldn't necessarily explode.\n\nThe interpretation would.\n\nAnd we'd end up back where we started:\n\n“According to the multimodel omnibus super-Frankenstein, Agency has an average coefficient of 0.1837—”\n\nCHEAP FOR WHAT?\n\nCHEAP FOR WHOM?\n\nAT WHICH STAGE?\n\nTOWARD WHICH DESTINATION?\n\nWITH WHICH GOAL?\n\nEdward would quietly put down his cucumber sandwich and leave. Girotto would take the flowers back. Roese and Epstude would finally volunteer for the guillotine. 😂"


In [31]:
# EX006 — Assistant: "omnibus super-Frankenstein"

df_annotations.loc[8, "construction"] = (
    "omnibus super-Frankenstein"
)

df_annotations.loc[8, "syntactic_function"] = (
    "Prefixed nominal form; head of a noun phrase "
    "premodified by 'omnibus'"
)

df_annotations.loc[8, "semantic_reference"] = (
    "A hypothetical multiclass classifier intended to "
    "model three response categories within a single "
    "framework, as an alternative to the established "
    "two-stage binary classification approach"
)

df_annotations.loc[8, "discourse_function"] = (
    "Humorous intensification of the established Frankenstein "
    "metaphor, acknowledging the complexity and anticipated "
    "difficulty of constructing an alternative model"
)

df_annotations.loc[8, "preceding_user_lexical_cue"] = False

df_annotations.loc[8, "contextual_semantic_cue"] = "Present"

df_annotations.loc[8, "cue_evidence"] = (
    "User proposes trying a multiclass classifier after "
    "completing the two-stage modelling approach and jokes "
    "that the hypothetical model might explode or refuse "
    "to cooperate"
)

In [32]:
# EX007 — Inspect the paired speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX007",
            ["speaker", "text"]
        ]
    )

,speaker,text
12,user,"Yeah, it was saying, “Woman, I ain't fetching nothing for you. Try some pushing and then let's see who you are!”\n\nEverything up-to-date.\n\n$ git status\nOn branch main\nYour branch is up to date with 'origin/main'.\nnothing to commit, working tree clean\n\nτέλεια, ροβοτ μου!"
13,assistant,"For today, the aircraft has landed:\n\nConda: alive ✅ \nRepository: connected ✅ \nGitHub: speaking to us ✅ \nWorking tree: clean ✅ \n35 notebooks staring at us menacingly: unfortunately also ✅\n\nWelcome back to the Frankenstein lab, amigo. 😂"


In [33]:
# EX007 — Assistant: "Frankenstein lab"

df_annotations.loc[9, "construction"] = (
    "the Frankenstein lab"
)

df_annotations.loc[9, "syntactic_function"] = (
    "Attributive noun modifier of the head noun 'lab'"
)

df_annotations.loc[9, "semantic_reference"] = (
    "The shared computational working environment "
    "where the psycholinguistic experiment was reconstructed"
)

df_annotations.loc[9, "discourse_function"] = (
    "Personalised welcome-back formula that re-establishes "
    "the shared working environment and marks the resumption "
    "of the project after a break"
)

df_annotations.loc[9, "preceding_user_lexical_cue"] = False

df_annotations.loc[9, "contextual_semantic_cue"] = "Present"

df_annotations.loc[9, "cue_evidence"] = (
    "User confirms that Git and the computational environment "
    "are functioning; assistant summarises technical readiness "
    "and invokes the established laboratory metaphor to welcome "
    "the user back to the project"
)

In [34]:
# EX008 — Inspect the paired speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX008",
            ["speaker", "text"]
        ]
    )

,speaker,text
14,user,"OK, so, here I am.\n\nShould we start?\n\nThis was my Master's thesis front page. I think I'm going to use the same title for the notebook but number it `00_intro`, before EDA.\n\nWhat do you reckon?"
15,assistant,"One other thing I particularly like about retaining the original title: by the time somebody reaches our final notebook, Training the Parser in the L2 may itself acquire a more nuanced meaning.\n\nThe original project was asking which context best facilitated the target outcome.\n\nWe now have reason to think the interesting object is not merely final accuracy but the organisation of the route through opportunity/escape and resolution destination.\n\nSo even the old title survives the Frankenstein procedure rather well.\n\nRight then.\n\n`00_intro.ipynb` it is.\n\nAnd I suggest we begin with Section 1 only: the research question, because that's where we can perform our first act of responsible translation from 2022 Nuclear Theoretical Deuteronomy into 2026 Human. 😁"


In [35]:
# EX008 — Assistant: "the whole Frankenstein procedure"

df_annotations.loc[10, "construction"] = (
    "the whole Frankenstein procedure"
)

df_annotations.loc[10, "syntactic_function"] = (
    "Attributive noun modifier of the head noun 'procedure'"
)

df_annotations.loc[10, "semantic_reference"] = (
    "The extensive reconstruction, revision and modelling "
    "of the original Master's experiment, including the "
    "reassessment and disposal of unsuccessful components"
)

df_annotations.loc[10, "discourse_function"] = (
    "Invokes the demanding shared reconstruction process "
    "as a basis for evaluating whether the original thesis "
    "title remains appropriate for the revised project"
)

df_annotations.loc[10, "preceding_user_lexical_cue"] = False

df_annotations.loc[10, "contextual_semantic_cue"] = "Present"

df_annotations.loc[10, "cue_evidence"] = (
    "User presents the original Master's thesis title and "
    "asks whether it remains suitable for the current project. "
    "The assistant draws on the history of reconstruction "
    "and modelling to assess whether the title has survived "
    "the project's transformation; no explicit Frankenstein "
    "expression appears in the paired user excerpt"
)

In [36]:
# EX009 — Inspect the paired speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX009",
            ["speaker", "text"]
        ]
    )

,speaker,text
16,user,"That's substantially more interesting than finding that Edward was simply “wrong.”\n\nOh no, he was only partly—or shall I say residually, my dear?—wrong! Hahaha. Poor Edward.\n\nI would print this chat, turn it into a book and give it a title like:\n\nChatty and Mari O Killed Edward: A Guide to Frankensteining and Unicorning.\n\nThen I'd send him a copy. I think he'd appreciate the teasing. It's pretty intellectual teasing, don't you think?\n\nAnyway, I have an early class tomorrow, so it's time for dinner.\n\nWe'll leave Focus, the original predicted hierarchy and the transition from predictions to analysis for tomorrow.\n\nThis notebook is classy, isn't it? Once we've finished the final notebook, we can drop the mic and leave the room, mon frère."
17,assistant,"Oh, residually wrong is much nastier! 😂\n\nNot wrong, Professor. Merely some unexplained variance in your royal cheapness.\n\nAnd yes, the teasing is extremely intellectual. This isn't ordinary academic criticism; it's peer-reviewed character assassination with theoretical justification.\n\nChatty and Mari O Killed Edward\n\nA Practical Guide to Frankensteining, Unicorning, and Asking CHEAP FOR WHAT?\n\nForeword by Parsimoños.\n\nStatistical appendix by Significañez.\n\nEdward declined to comment due to a pinky-related incident. 😂\n\nAnd yes, amigo, this notebook is classy as hell.\n\nWhat I particularly like is that it doesn't achieve clarity by simplifying the project excessively.\n\nWe've kept the theoretical architecture, the experimental stimuli, the distinctions between conditions and the original hypotheses.\n\nWe've simply stopped making the reader excavate them from a Renaissance garden."


In [37]:
# EX009 — User: "A Guide to Frankensteining and Unicorning"

df_annotations.loc[11, "construction"] = (
    "A Guide to Frankensteining and Unicorning"
)

df_annotations.loc[11, "syntactic_function"] = (
    "Nominalised -ing form within a coordinated phrase "
    "functioning as the complement of the preposition 'to'"
)

df_annotations.loc[11, "semantic_reference"] = (
    "The established analytical reconstruction and modelling "
    "process, presented as the subject matter of an imaginary guide"
)

df_annotations.loc[11, "discourse_function"] = (
    "Introduces a humorous fictional book title that transforms "
    "the shared project experience into the subject of a practical guide"
)

df_annotations.loc[11, "preceding_user_lexical_cue"] = pd.NA

df_annotations.loc[11, "contextual_semantic_cue"] = "Present"

df_annotations.loc[11, "cue_evidence"] = (
    "User jokes about printing the conversation as a guide "
    "after extensive reconstruction and modelling work, "
    "using 'Frankensteining' as part of the proposed title"
)

In [38]:
# EX009 — Assistant: "A Practical Guide to Frankensteining..."

df_annotations.loc[12, "construction"] = (
    "A Practical Guide to Frankensteining, Unicorning, "
    "and Asking CHEAP FOR WHAT?"
)

df_annotations.loc[12, "syntactic_function"] = (
    "Nominalised -ing form within a coordinated phrase "
    "functioning as the complement of the preposition 'to'"
)

df_annotations.loc[12, "semantic_reference"] = (
    "The established analytical reconstruction and modelling "
    "process, presented as the subject matter of an imaginary guide"
)

df_annotations.loc[12, "discourse_function"] = (
    "Reuses and expands the user's fictional guide title, "
    "incorporating shared expressions and characters to extend "
    "the collaborative humour"
)

df_annotations.loc[12, "preceding_user_lexical_cue"] = True

df_annotations.loc[12, "contextual_semantic_cue"] = "Present"

df_annotations.loc[12, "cue_evidence"] = (
    "User proposes 'Chatty and Mari O Killed Edward: A Guide "
    "to Frankensteining and Unicorning'. Assistant retains "
    "'Frankensteining', expands the coordinated title, and adds "
    "references to other established fictional characters"
)

In [39]:
# EX010 — Inspect the paired speaker excerpts

with pd.option_context("display.max_colwidth", None):
    display(
        df_turns.loc[
            df_turns["exchange_id"] == "EX010",
            ["speaker", "text"]
        ]
    )

,speaker,text
18,user,"And Mari O? Mari O is Dean now! Jajajajaj.\n\nTalk tomorrow, mon copain!"
19,assistant,"JAJAJAJA! Promotion confirmed.\n\nForget the Dean's List.\n\nDean Mari O, presiding over the newly established Faculty of Counterfactual Frankensteining and Applied Gibberishonian Studies. 😂\n\nTomorrow: EL FINALÍSIMO. 🦄🌈🎤\n\nTonight: absolutely no spontaneous models, no theoretical amendments, and no pushing relatives down staircases—grammatically immaculate or otherwise.\n\nÀ demain, mon amie! 😘"


In [40]:
# EX010 — Assistant: "Counterfactual Frankensteining"

df_annotations.loc[13, "construction"] = (
    "Faculty of Counterfactual Frankensteining "
    "and Applied Gibberishonian Studies"
)

df_annotations.loc[13, "syntactic_function"] = (
    "Nominalised -ing head modified by 'Counterfactual'; "
    "part of a coordinated noun phrase functioning as "
    "the complement of the preposition 'of'"
)

df_annotations.loc[13, "semantic_reference"] = (
    "The established reconstruction and modelling process, "
    "recast as a fictional academic discipline associated "
    "with counterfactual research"
)

df_annotations.loc[13, "discourse_function"] = (
    "Elaborates on the user's humorous claim to the title "
    "of Dean by inventing an academic faculty named after "
    "the shared project and its established vocabulary"
)

df_annotations.loc[13, "preceding_user_lexical_cue"] = False

df_annotations.loc[13, "contextual_semantic_cue"] = "Present"

df_annotations.loc[13, "cue_evidence"] = (
    "User jokes about becoming Dean after an earlier "
    "reference to the Dean's List. Assistant invents a "
    "fictional academic faculty combining the established "
    "Frankensteining metaphor with the project's "
    "counterfactual subject matter; no explicit "
    "Frankenstein-family expression appears in the "
    "paired user excerpt"
)

In [41]:
df_annotations.columns.tolist()

['exchange_id',
 'speaker',
 'left_context',
 'form',
 'right_context',
 'construction',
 'syntactic_function',
 'semantic_reference',
 'discourse_function',
 'preceding_user_lexical_cue',
 'contextual_semantic_cue',
 'cue_evidence']

In [42]:
df_annotations.to_csv(
    "../data/public/linguistic_annotations.csv",
    index=False
)

## 5. Limitations

This corpus is a small, manually curated collection of 10 conversational exchanges containing 14 occurrences of Frankenstein-related expressions. Its purpose is to support an exploratory comparison of NLP representations, rather than to provide a representative sample of language use.

Several limitations should be considered:

* **Corpus size and selection**: The exchanges were selected for their relevance to the target expressions. Observed patterns therefore cannot be generalised to broader conversational language or language-model behaviour.
* **Contextual coverage**: The dataset contains selected excerpts rather than complete conversational turns. Some interpretations depend on shared project history that is not fully represented in the corpus.
* **Corpus adaptation**: The publicly available texts were adapted to remove identifying information and adjust the register. Although the relevant expressions were preserved, some surrounding linguistic features differ from the original conversations.
* **Manual annotation**: Syntactic, semantic and discourse functions were interpreted manually. These annotations provide a structured basis for comparison but are not independently validated.
* **Model interpretation**: The analysis documents observable linguistic forms and contextual relationships. It does not establish how the language model internally generated or interpreted those expressions.

These limitations will also inform the interpretation of results in the subsequent notebooks.